# 00 - EDA y Calidad de Datos\n\nExploración inicial del archivo `venta_tiendas.csv` y revisión de calidad para definir reglas de transformación en Silver.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab usando Google Drive personal.

Estructura esperada en Drive:

```text
Proyecto_BigData_Forus/
├── raw/
│   └── venta_tiendas.csv
├── bronze/
├── silver/
├── gold/
└── evidencias/
```


In [ ]:
# Instalación de dependencias para Colab
!pip install -q pyspark==3.5.1 delta-spark==3.2.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
print(f"Python Version: {sys.version}")

# Uninstall conflicting packages and ensure pyspark and delta-spark are installed
!pip uninstall -y dataproc-spark-connect
!pip uninstall -y opentelemetry-api
!pip uninstall -y importlib-metadata
!pip install -q importlib-metadata==8.0.0

# Explicitly uninstall pyspark and delta-spark before reinstalling to ensure clean slate
!pip uninstall -y pyspark delta-spark

# Install compatible versions for Python 3.12
!pip install pyspark==3.4.1 delta-spark==2.4.0

import os
import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Correcting RUTA_BASE to point to the project directory
# Based on the expected structure: Proyecto_BigData_Forus/raw/venta_tiendas.csv
RUTA_BASE = "/content/drive/MyDrive/Proyecto_BigData_Forus"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

ARCHIVO_VENTAS = "venta_tiendas.csv"

for ruta in [RUTA_RAW, RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_EVIDENCIAS]:
    os.makedirs(ruta, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Ruta base:", RUTA_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Found existing installation: importlib_metadata 8.0.0
Uninstalling importlib_metadata-8.0.0:
  Successfully uninstalled importlib_metadata-8.0.0
  Using cached pyspark-3.4.1.tar.gz (310.8 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.1-py2.py3-none-any.whl size=311285391 sha256=cefb0f5e845fc497fe0ad2fc9a1db74aedb4347c8ce7be6e3285ab1a8a0fdd81
  Stored in directory: /root/.cache/pip/wheels/8d/95/1d/739a17bda5d6a1c3c6f60eed9a82f600ab0d9fcd4c601ce0da
Successfully built pyspark
Spark: 3.4.1
Ruta base: /content/drive/MyDrive/Proyecto_BigData_Forus


## 1. Lectura del CSV desde Google Drive\n\nEl archivo debe estar en `Proyecto_BigData_Forus/raw/venta_tiendas.csv`.

In [4]:

ruta_csv = "/content/drive/MyDrive/venta_tiendas.csv" #  path

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(ruta_csv)
)

print("Registros:", df.count())
print("Columnas:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)


Registros: 2250970
Columnas: 11
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: integer (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: integer (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: integer (nullable = true)
 |-- costo: integer (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion        |cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|1       |1921

## 2. Revisión de valores nulos

In [5]:
from pyspark.sql.functions import col, sum as spark_sum, when

nulos = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

nulos.show(truncate=False)


+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion|cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+
|0       |0                 |0         |0            |0                |0                     |0             |0          |0       |0    |0    |
+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+



## 3. Revisión de duplicados por transacción y producto

In [6]:
claves = ["id_canal", "numero_transaccion", "numero_pos", "numero_boleta", "id_producto"]

total = df.count()
distintos = df.select(claves).distinct().count()

print("Total registros:", total)
print("Combinaciones únicas por clave:", distintos)
print("Posibles duplicados:", total - distintos)


Total registros: 2250970
Combinaciones únicas por clave: 2246621
Posibles duplicados: 4349


## 4. Estadísticas descriptivas de variables numéricas

In [7]:
columnas_numericas = ["unidades", "venta", "costo"]

df.select(columnas_numericas).describe().show()


+-------+------------------+-----------------+------------------+
|summary|          unidades|            venta|             costo|
+-------+------------------+-----------------+------------------+
|  count|           2250970|          2250970|           2250970|
|   mean|0.8490846168540673|23822.78605667779| 9772.421610239142|
| stddev|0.6309010987041614|36437.25825146752|13480.440706964917|
|    min|               -45|         -2646681|           -842708|
|    max|                85|         16070786|           4894690|
+-------+------------------+-----------------+------------------+



## 5. Revisión del formato de fecha\n\nEn el archivo original la fecha viene en la columna `fecha_transaccion`, por ejemplo `14/03/2016 12:00:00 AM CL`. En Bronze se conservará igual y en Silver se transformará a `fecha_venta` estándar.

In [8]:
df.select("fecha_transaccion").distinct().show(10, truncate=False)


+-------------------------+
|fecha_transaccion        |
+-------------------------+
|07/07/2023 12:00:00 AM CL|
|20/06/2020 12:00:00 AM CL|
|17/11/2017 12:00:00 AM CL|
|26/03/2022 12:00:00 AM CL|
|30/01/2016 12:00:00 AM CL|
|25/11/2015 12:00:00 AM CL|
|17/05/2018 12:00:00 AM CL|
|15/04/2015 12:00:00 AM CL|
|08/01/2026 12:00:00 AM CL|
|04/07/2019 12:00:00 AM CL|
+-------------------------+
only showing top 10 rows



## 6. Métricas base para evidencia

In [9]:
from pyspark.sql.functions import countDistinct, min as spark_min, max as spark_max

metricas = df.agg(
    countDistinct("numero_boleta").alias("boletas_distintas"),
    countDistinct("id_producto").alias("productos_distintos"),
    countDistinct("cod_tienda_facturacion").alias("tiendas_distintas"),
    spark_min("fecha_transaccion").alias("fecha_min_raw"),
    spark_max("fecha_transaccion").alias("fecha_max_raw")
)

metricas.show(truncate=False)


+-----------------+-------------------+-----------------+-------------------------+-------------------------+
|boletas_distintas|productos_distintos|tiendas_distintas|fecha_min_raw            |fecha_max_raw            |
+-----------------+-------------------+-----------------+-------------------------+-------------------------+
|865065           |420699             |469              |01/02/2023 12:00:00 AM CL|31/10/2025 12:00:00 AM CL|
+-----------------+-------------------+-----------------+-------------------------+-------------------------+

